# 06 — Expérimentation et entraînement des modèles

## Objectif

Cette étape vise à expérimenter différentes approches de Machine Learning afin de prévoir la consommation électrique des sept sites EnerVision sur les **24 prochaines heures**, avec une granularité horaire.

Les expérimentations s'appuient sur les datasets analytiques construits lors de la phase de préparation, à partir de 30 jours consécutifs de mesures énergétiques simulées.

L'objectif est de déterminer quelles stratégies de modélisation permettent de représenter au mieux les comportements énergétiques des sites, notamment les variations horaires, les cycles hebdomadaires et les différences liées à leur activité.

## Stratégie d'expérimentation

Deux approches principales seront étudiées :

- **Modèle global :** un modèle unique entraîné sur les données des sept sites, en intégrant leurs caractéristiques comme variables explicatives.
- **Modèles spécialisés :** des modèles distincts entraînés sur les données de chaque type de site (bureaux, usines, datacenter, commerce et hôpital).

Une méthode de référence (*baseline*) sera également mise en place afin de disposer d'un point de comparaison simple et mesurable.

Les différentes approches seront évaluées sur des périodes temporelles comparables, afin de déterminer si la spécialisation des modèles apporte une amélioration réelle des performances prédictives.

## Méthodologie

Le travail d'expérimentation sera organisé autour des étapes suivantes :

1. **Chargement et validation :** import des datasets préparés et vérification de leur intégrité.
2. **Séparation chronologique :** définition des périodes d'entraînement et de validation, en tenant compte des horizons de prévision H+1 à H+24.
3. **Prétraitement ML :** sélection des variables, traitement des valeurs manquantes et encodage des caractéristiques catégorielles.
4. **Baseline :** mise en place d'une méthode de prévision simple servant de référence.
5. **Entraînement :** expérimentation de plusieurs algorithmes de régression.
6. **Comparaison :** évaluation des modèles globaux et spécialisés sur les mêmes observations de validation.
7. **Traçabilité :** enregistrement des paramètres, des métriques et des résultats dans MLflow.

Les performances seront notamment évaluées à l'aide de la **MAE** (*Mean Absolute Error*) et de la **RMSE** (*Root Mean Squared Error*).

Les résultats seront également analysés par site, par type d'activité et par horizon de prévision, afin de ne pas limiter l'évaluation à une métrique globale.

## Prévention des fuites de données

Les observations supervisées sont construites à partir de séries temporelles dont les fenêtres et les horizons de prévision se chevauchent.

Une séparation aléatoire des exemples risquerait donc d'introduire des informations futures dans les données d'entraînement.

La validation reposera sur une séparation chronologique explicite, garantissant que les consommations utilisées comme cibles d'entraînement précèdent la période de validation.

Les transformations apprises sur les données, notamment l'imputation et l'encodage, seront ajustées exclusivement sur l'ensemble d'entraînement.

## Limites de l'expérimentation

L'historique disponible couvre 30 jours et provient d'une API simulée. Il permet de développer et de comparer les premières stratégies de prévision, mais ne constitue pas une base suffisante pour démontrer leur robustesse à long terme.

Les résultats devront être consolidés sur des historiques plus étendus et, à terme, sur des mesures réelles issues de l'API current.

Cette phase vise donc avant tout à valider une démarche expérimentale reproductible, avant son industrialisation dans le pipeline Machine Learning de Smart Energy Optimizer.

## 1 — Chargement des datasets

### A - Imports et configuration

In [353]:
from pathlib import Path

import numpy as np
import pandas as pd

### B - Chargement des fichiers Parquet

In [354]:
DATA_DIR = Path("../data/processed")

hourly_path = DATA_DIR / "hourly_dataset.parquet"
training_path = DATA_DIR / "training_dataset.parquet"

assert hourly_path.is_file(), f"Fichier introuvable : {hourly_path}"
assert training_path.is_file(), f"Fichier introuvable : {training_path}"

df_hourly = pd.read_parquet(hourly_path)
df_training = pd.read_parquet(training_path)

print("Dataset horaire :", df_hourly.shape)
print("Dataset supervisé :", df_training.shape)

display(df_training.head())

Dataset horaire : (5040, 20)
Dataset supervisé : (118860, 14)


,forecast_origin,target_timestamp,site_id,site_type,horizon,target_hour,target_day_of_week,target_month,target_is_weekend,lag_1h,lag_24h,rolling_mean_24h,target_lag_168h,target_consumption_kw
0,2026-09-24 00:00:00+00:00,2026-09-24 01:00:00+00:00,SITE001,office,1,3,3,9,0,NaN,NaN,NaN,NaN,62.300833
1,2026-09-24 01:00:00+00:00,2026-09-24 02:00:00+00:00,SITE001,office,1,4,3,9,0,60.726364,NaN,NaN,NaN,62.099000
2,2026-09-24 02:00:00+00:00,2026-09-24 03:00:00+00:00,SITE001,office,1,5,3,9,0,62.300833,NaN,NaN,NaN,61.398182
3,2026-09-24 03:00:00+00:00,2026-09-24 04:00:00+00:00,SITE001,office,1,6,3,9,0,62.099000,NaN,NaN,NaN,60.970909
4,2026-09-24 04:00:00+00:00,2026-09-24 05:00:00+00:00,SITE001,office,1,7,3,9,0,61.398182,NaN,NaN,NaN,62.373333


## 2 - Vérification de l'intégrité des datasets

### A - Structure et cohérence temporelle

In [355]:
# Dimensions attendues pour notre collecte mensuelle
assert len(df_hourly) == 30 * 24 * 7
assert len(df_training) == 118_860

# Unicité des observations horaires
assert not df_hourly.duplicated(subset=["site_id", "timestamp_hour"]).any()

# Cohérence des horizons
assert df_training["horizon"].between(1, 24).all()

assert (
    df_training["target_timestamp"]
    == df_training["forecast_origin"] + pd.to_timedelta(df_training["horizon"], unit="h")
).all()

# Unicité des exemples supervisés
assert not df_training.duplicated(subset=["site_id", "forecast_origin", "horizon"]).any()

print("Intégrité structurelle et temporelle validée.")

Intégrité structurelle et temporelle validée.


### B -Couverture temporelle et répartition par site

In [356]:
print("Période couverte par les observations horaires :")
print(df_hourly["timestamp_hour"].min())
print(df_hourly["timestamp_hour"].max())

print("\nPériode couverte par les exemples supervisés :")
print(df_training["forecast_origin"].min())
print(df_training["target_timestamp"].max())

print("\nRépartition des observations horaires par site :")
display(df_hourly.groupby(["site_id", "site_type"]).size().to_frame("observations"))

print("\nRépartition des exemples supervisés par type de site :")
display(
    df_training.groupby("site_type").agg(
        observations=("horizon", "size"),
        sites=("site_id", "nunique"),
    )
)

Période couverte par les observations horaires :
2026-09-24 00:00:00+00:00
2026-10-23 23:00:00+00:00

Période couverte par les exemples supervisés :
2026-09-24 00:00:00+00:00
2026-10-23 23:00:00+00:00

Répartition des observations horaires par site :


,,observations
site_id,site_type,
SITE001,office,720
SITE002,factory,720
SITE003,datacenter,720
SITE004,retail,720
SITE005,hospital,720
SITE006,office,720
SITE007,factory,720



Répartition des exemples supervisés par type de site :


,observations,sites
site_type,,
datacenter,16980,1
factory,33960,2
hospital,16980,1
office,33960,2
retail,16980,1


### C - Analyse des valeurs manquantes

In [357]:
missing = pd.DataFrame(
    {
        "missing_count": df_training.isna().sum(),
        "missing_percent": (df_training.isna().mean() * 100).round(2),
    }
)

display(
    missing.sort_values(
        "missing_count",
        ascending=False,
    )
)

,missing_count,missing_percent
target_lag_168h,26124,21.98
lag_24h,4032,3.39
rolling_mean_24h,3024,2.54
lag_1h,168,0.14
forecast_origin,0,0.00
target_timestamp,0,0.00
target_hour,0,0.00
horizon,0,0.00
site_type,0,0.00
site_id,0,0.00


## 3 — Séparation chronologique des données

### Objectif

La séparation des données vise à constituer deux ensembles distincts : un ensemble d'entraînement permettant aux modèles d'apprendre les comportements énergétiques et un ensemble de validation destiné à évaluer leurs performances sur des périodes ultérieures.

Contrairement à une régression classique sur des observations indépendantes, notre dataset supervisé contient plusieurs horizons de prévision construits à partir des mêmes séries temporelles.

Une séparation aléatoire risquerait donc d'introduire des fuites d'information entre les ensembles.

### Stratégie retenue

Une première séparation chronologique est définie sur les 30 jours disponibles :

- **21 jours pour l'entraînement**, du 24 septembre au 14 octobre 2026.
- **9 jours pour la validation**, du 15 au 23 octobre 2026.

Pour garantir l'intégrité temporelle, les exemples d'entraînement devront avoir une heure cible strictement antérieure au début de la validation.

Les exemples de validation seront sélectionnés à partir de leur instant de prévision, afin de reproduire une situation dans laquelle le modèle prédit les consommations futures à partir des informations disponibles à cet instant.

Cette stratégie sera commune aux modèles globaux et spécialisés afin de garantir la comparabilité des résultats.

### Limites

Cette première séparation permet une évaluation expérimentale cohérente, mais elle ne garantit pas la représentativité de toutes les conditions d'exploitation.

Des validations sur plusieurs fenêtres chronologiques pourront être envisagées ultérieurement afin de renforcer la robustesse des comparaisons.

### A - Construction des ensembles d'entraînements et de validation

In [358]:
# Date de séparation chronologique (UTC)
SPLIT_DATE = pd.Timestamp("2026-10-15", tz="UTC")

# Entraînement : seules les consommations antérieures
# à la date de séparation peuvent servir de cibles.
df_train = df_training.loc[df_training["target_timestamp"] < SPLIT_DATE].copy()

# Validation : prévisions effectuées à partir
# de la date de séparation.
df_val = df_training.loc[df_training["forecast_origin"] >= SPLIT_DATE].copy()

print(f"Exemples d'entraînement : {len(df_train):,}")
print(f"Exemples de validation : {len(df_val):,}")

print("\nPériode d'entraînement :")
print("Première origine :", df_train["forecast_origin"].min())
print("Dernière cible :", df_train["target_timestamp"].max())

print("\nPériode de validation :")
print("Première origine :", df_val["forecast_origin"].min())
print("Dernière cible :", df_val["target_timestamp"].max())

Exemples d'entraînement : 82,572
Exemples de validation : 34,188

Période d'entraînement :
Première origine : 2026-09-24 00:00:00+00:00
Dernière cible : 2026-10-14 23:00:00+00:00

Période de validation :
Première origine : 2026-10-15 00:00:00+00:00
Dernière cible : 2026-10-23 23:00:00+00:00


### B - Véricication de l'intégrité temporelle

In [359]:
# Aucune cible d'entraînement ne doit atteindre
# la période de validation.
assert (df_train["target_timestamp"] < SPLIT_DATE).all()

# Aucune prévision de validation ne doit être
# effectuée avant le début de la validation.
assert (df_val["forecast_origin"] >= SPLIT_DATE).all()

# Aucun chevauchement entre les cibles d'entraînement
# et les origines de validation.
assert df_train["target_timestamp"].max() < df_val["forecast_origin"].min()

# Les deux ensembles doivent contenir les sept sites.
assert df_train["site_id"].nunique() == 7
assert df_val["site_id"].nunique() == 7

print("Séparation chronologique validée.")

Séparation chronologique validée.


### C - Répartition des exemples par type de site

In [360]:
distribution = pd.DataFrame(
    {
        "train": df_train.groupby("site_type").size(),
        "validation": df_val.groupby("site_type").size(),
    }
)

display(distribution)

,train,validation
site_type,,
datacenter,11796,4884
factory,23592,9768
hospital,11796,4884
office,23592,9768
retail,11796,4884


### D - Validation des horizons disponibles

In [361]:
horizon_distribution = pd.DataFrame(
    {
        "train": df_train.groupby("horizon").size(),
        "validation": df_val.groupby("horizon").size(),
    }
)

display(horizon_distribution)

assert set(df_train["horizon"].unique()) == set(range(1, 25))
assert set(df_val["horizon"].unique()) == set(range(1, 25))

print("Les 24 horizons sont représentés dans les deux ensembles.")

,train,validation
horizon,,
1,3521,1505
2,3514,1498
3,3507,1491
4,3500,1484
5,3493,1477
6,3486,1470
7,3479,1463
8,3472,1456
9,3465,1449


Les 24 horizons sont représentés dans les deux ensembles.


### Bilan de la séparation chronologique

La séparation chronologique a permis de constituer deux ensembles distincts :

- **Entraînement :** 82 572 exemples, dont les cibles précèdent le 15 octobre 2026.
- **Validation :** 34 188 exemples, correspondant aux prévisions effectuées à partir du 15 octobre 2026.

Les contrôles ont confirmé l'absence de chevauchement temporel entre les cibles d'entraînement et les origines de validation.

Les sept sites et les cinq catégories d'activité sont représentés dans les deux ensembles.

Les exemples dont l'origine précède la séparation mais dont la cible appartient à la période de validation ont été exclus afin de préserver l'intégrité temporelle.

Cette séparation constitue une première base commune pour comparer les modèles globaux et spécialisés.

## 4 — Prétraitement des données pour le Machine Learning

### Objectif

Cette étape vise à préparer les ensembles d'entraînement et de validation afin de les rendre exploitables par les algorithmes de Machine Learning.

Les données issues de la phase de préparation contiennent des variables numériques, des caractéristiques catégorielles ainsi que certaines valeurs manquantes dans les variables historiques.

L'objectif est de définir une chaîne de transformation cohérente, reproductible et compatible avec les différentes stratégies de modélisation envisagées.

### Stratégie de prétraitement

Le prétraitement reposera sur les opérations suivantes :

- **Séparation des variables :** distinction entre les caractéristiques explicatives (`X`) et la consommation électrique à prédire (`y`).
- **Traitement des valeurs manquantes :** application d'une stratégie d'imputation aux variables historiques, sans modifier les valeurs cibles.
- **Encodage des variables catégorielles :** transformation des identifiants et types de sites en caractéristiques numériques exploitables par les modèles.
- **Construction d'un pipeline :** utilisation des composants `Pipeline` et `ColumnTransformer` de scikit-learn pour centraliser les transformations.

### Prévention des fuites de données

Les paramètres des transformations, notamment les valeurs utilisées pour l'imputation, seront déterminés exclusivement à partir de l'ensemble d'entraînement.

Le pipeline ainsi ajusté sera ensuite appliqué à l'ensemble de validation sans nouvel apprentissage.

Cette approche garantit que les transformations ne tirent aucune information statistique des données de validation.

### Perspectives

Le prétraitement sera initialement défini pour un modèle global exploitant les données des sept sites.

Il pourra ensuite être adapté aux modèles spécialisés par type d'activité, afin de garantir une comparaison cohérente entre les différentes approches.

À terme, ces transformations seront intégrées au pipeline d'inférence pour assurer la cohérence entre les données utilisées lors de l'entraînement et celles reçues en production.

### A - Séparation des variables explicatives et de la cible

In [362]:
feature_columns = [
    "site_id",
    "site_type",
    "horizon",
    "target_hour",
    "target_day_of_week",
    "target_month",
    "target_is_weekend",
    "lag_1h",
    "lag_24h",
    "rolling_mean_24h",
    "target_lag_168h",
]

target_column = "target_consumption_kw"

# Ensemble d'entraînement
X_train = df_train[feature_columns].copy()
y_train = df_train[target_column].copy()

# Ensemble de validation
X_val = df_val[feature_columns].copy()
y_val = df_val[target_column].copy()

print("Entraînement :")
print("X_train :", X_train.shape)
print("y_train :", y_train.shape)

print("\nValidation :")
print("X_val :", X_val.shape)
print("y_val :", y_val.shape)

Entraînement :
X_train : (82572, 11)
y_train : (82572,)

Validation :
X_val : (34188, 11)
y_val : (34188,)


### B - Contrôle des variables

In [363]:
assert X_train.columns.tolist() == X_val.columns.tolist()

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)

assert y_train.notna().all()
assert y_val.notna().all()

print("Séparation des variables validée.")

print("\nTypes des variables :")
display(X_train.dtypes.to_frame("dtype"))

print("\nValeurs manquantes dans les variables explicatives :")
display(
    pd.DataFrame(
        {
            "train": X_train.isna().sum(),
            "validation": X_val.isna().sum(),
        }
    )
)

Séparation des variables validée.

Types des variables :


,dtype
site_id,str
site_type,str
horizon,int64
target_hour,int32
target_day_of_week,int32
target_month,int32
target_is_weekend,int64
lag_1h,float64
lag_24h,float64
rolling_mean_24h,float64



Valeurs manquantes dans les variables explicatives :


,train,validation
site_id,0,0
site_type,0,0
horizon,0,0
target_hour,0,0
target_day_of_week,0,0
target_month,0,0
target_is_weekend,0,0
lag_1h,168,0
lag_24h,4032,0
rolling_mean_24h,3024,0


### C - Construction du pipeline de prétraitement

#### C1 - Définition des transformations

In [364]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Variables numériques sans valeurs manquantes
numeric_columns = [
    "horizon",
    "target_hour",
    "target_day_of_week",
    "target_month",
    "target_is_weekend",
]

# Variables historiques pouvant contenir des NaN
historical_columns = [
    "lag_1h",
    "lag_24h",
    "rolling_mean_24h",
    "target_lag_168h",
]

# Variables catégorielles
categorical_columns = [
    "site_id",
    "site_type",
]

# Traitement des variables historiques
historical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

# Prétraitement global
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numeric_columns),
        ("historical", historical_pipeline, historical_columns),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns,
        ),
    ],
    remainder="drop",
)

#### C2 - Apprentissage et application des transformations

In [365]:
# Apprentissage des transformations sur l'entraînement uniquement
X_train_prepared = preprocessor.fit_transform(X_train)

# Application des mêmes transformations à la validation
X_val_prepared = preprocessor.transform(X_val)

print("Entraînement :", X_train_prepared.shape)
print("Validation :", X_val_prepared.shape)

print("\nVariables après transformation :")
print(preprocessor.get_feature_names_out())

Entraînement : (82572, 21)
Validation : (34188, 21)

Variables après transformation :
['numeric__horizon' 'numeric__target_hour' 'numeric__target_day_of_week'
 'numeric__target_month' 'numeric__target_is_weekend' 'historical__lag_1h'
 'historical__lag_24h' 'historical__rolling_mean_24h'
 'historical__target_lag_168h' 'categorical__site_id_SITE001'
 'categorical__site_id_SITE002' 'categorical__site_id_SITE003'
 'categorical__site_id_SITE004' 'categorical__site_id_SITE005'
 'categorical__site_id_SITE006' 'categorical__site_id_SITE007'
 'categorical__site_type_datacenter' 'categorical__site_type_factory'
 'categorical__site_type_hospital' 'categorical__site_type_office'
 'categorical__site_type_retail']


#### C3 - Vérification du résultat

In [366]:
assert X_train_prepared.shape[0] == len(y_train)
assert X_val_prepared.shape[0] == len(y_val)

assert np.isfinite(X_train_prepared).all()
assert np.isfinite(X_val_prepared).all()

print("Prétraitement validé : aucune valeur manquante ou infinie.")

Prétraitement validé : aucune valeur manquante ou infinie.


## 5 — Construction et évaluation d'une baseline

### Objectif

Avant d'entraîner des modèles de Machine Learning, il est nécessaire de définir une méthode de référence (*baseline*) permettant d'évaluer l'intérêt réel des approches prédictives envisagées.

Une baseline repose sur une règle de prévision simple, sans apprentissage statistique complexe. Elle constitue un point de comparaison pour déterminer si les modèles entraînés apportent une amélioration mesurable.

### Stratégie retenue

Dans le cadre de la prévision énergétique, nous retenons une approche fondée sur la **persistance saisonnière**.

Le principe consiste à supposer que la consommation électrique d'un site à une heure donnée sera similaire à celle observée à la même heure la veille.

Cette méthode est pertinente pour notre cas d'étude, car l'analyse exploratoire a mis en évidence des comportements énergétiques récurrents, notamment des variations selon les heures, les jours de la semaine et les types de sites.

Elle présente néanmoins certaines limites, notamment lors des transitions entre jours ouvrés et week-ends.

### Méthodologie

La baseline sera évaluée sur l'ensemble de validation défini précédemment.

Pour chaque observation, nous utiliserons une consommation historique disponible au moment de la prévision, sans exploiter de données futures.

Les performances seront mesurées à l'aide de deux indicateurs :

- **MAE (Mean Absolute Error) :** erreur absolue moyenne, exprimée en kW.
- **RMSE (Root Mean Squared Error) :** racine de l'erreur quadratique moyenne, également exprimée en kW, qui pénalise davantage les erreurs importantes.

Les résultats seront analysés globalement, puis par type de site et par horizon de prévision.

### Finalité

Cette première évaluation constituera notre référence expérimentale.

Les modèles de Machine Learning développés par la suite devront démontrer leur capacité à améliorer ces résultats, dans des conditions de validation comparables.

### A - Construction des prédictions de référence

#### A1 - Construction de la baseline journalière

In [367]:
# Série des consommations horaires observées
hourly_consumption = df_hourly.set_index(["site_id", "timestamp_hour"])["consumption_kw"]

# Même heure la veille de l'heure cible
df_baseline = df_val[
    [
        "site_id",
        "site_type",
        "forecast_origin",
        "target_timestamp",
        "horizon",
        "target_consumption_kw",
    ]
].copy()

df_baseline["reference_timestamp"] = df_baseline["target_timestamp"] - pd.Timedelta(hours=24)

# Récupération de la consommation de référence
reference_index = pd.MultiIndex.from_frame(df_baseline[["site_id", "reference_timestamp"]])

df_baseline["baseline_prediction"] = hourly_consumption.reindex(reference_index).to_numpy()

# Une mesure de référence ne peut être utilisée
# que si elle est connue à l'instant de prévision.
df_baseline["reference_available"] = (
    df_baseline["reference_timestamp"] <= df_baseline["forecast_origin"]
) & df_baseline["baseline_prediction"].notna()

print("Disponibilité de la référence :")
display(df_baseline.groupby("horizon")["reference_available"].agg(["sum", "count", "mean"]))

Disponibilité de la référence :


,sum,count,mean
horizon,,,
1,1505,1505,1.0
2,1498,1498,1.0
3,1491,1491,1.0
4,1484,1484,1.0
5,1477,1477,1.0
6,1470,1470,1.0
7,1463,1463,1.0
8,1456,1456,1.0
9,1449,1449,1.0


#### A2 - Evaluation de la baseline journalière
 - Calcul des métriques
 - Analyse par type de site
 - Analyse par horizon de prévision

In [368]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# On conserve uniquement les références disponibles
df_baseline_valid = df_baseline.loc[df_baseline["reference_available"]].copy()

# Consommations réelles et prédites
y_true = df_baseline_valid["target_consumption_kw"]
y_pred = df_baseline_valid["baseline_prediction"]

# Calcul des métriques
baseline_mae = mean_absolute_error(y_true, y_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("Baseline journalière (H-24)")
print(f"Observations évaluées : {len(df_baseline_valid):,}")
print(f"MAE  : {baseline_mae:.2f} kW")
print(f"RMSE : {baseline_rmse:.2f} kW")

Baseline journalière (H-24)
Observations évaluées : 34,188
MAE  : 36.49 kW
RMSE : 114.25 kW


In [369]:
# Analyse par type de site
baseline_by_type = (
    df_baseline_valid.groupby("site_type")
    .apply(
        lambda group: pd.Series(
            {
                "observations": len(group),
                "MAE": mean_absolute_error(
                    group["target_consumption_kw"],
                    group["baseline_prediction"],
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        group["target_consumption_kw"],
                        group["baseline_prediction"],
                    )
                ),
            }
        ),
        include_groups=False,
    )
    .round(2)
)

display(baseline_by_type)

,observations,MAE,RMSE
site_type,,,
datacenter,4884.0,12.22,15.44
factory,9768.0,92.73,207.28
hospital,4884.0,7.98,10.10
office,9768.0,10.66,26.35
retail,4884.0,28.41,60.97


In [370]:
# Analyse par horizon de prévision
baseline_by_horizon = (
    df_baseline_valid.groupby("horizon")
    .apply(
        lambda group: pd.Series(
            {
                "MAE": mean_absolute_error(
                    group["target_consumption_kw"],
                    group["baseline_prediction"],
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        group["target_consumption_kw"],
                        group["baseline_prediction"],
                    )
                ),
            }
        ),
        include_groups=False,
    )
    .round(2)
)

display(baseline_by_horizon)

,MAE,RMSE
horizon,,
1,34.92,111.19
2,35.06,111.44
3,35.19,111.70
4,35.34,111.97
5,35.48,112.23
6,35.63,112.50
7,35.78,112.76
8,35.92,113.03
9,36.04,113.30


### Bilan de la baseline journalière

La première méthode de référence repose sur une persistance journalière : la consommation prévue pour une heure donnée correspond à celle observée à la même heure la veille.

L'évaluation a été réalisée sur les **34 188 exemples de validation**, pour lesquels les références historiques étaient disponibles.

Les résultats globaux sont les suivants :

- **MAE : 36,49 kW**
- **RMSE : 114,25 kW**

L'analyse par type de site révèle des écarts de performance importants. La méthode obtient des erreurs relativement faibles pour les hôpitaux, les bureaux et les datacenters, tandis que les usines présentent des erreurs nettement plus élevées.

Ces différences peuvent notamment s'expliquer par les variations de consommation propres aux activités industrielles et par les transitions entre jours ouvrés et week-ends. Cette hypothèse devra être vérifiée par des analyses complémentaires.

L'erreur moyenne augmente également légèrement selon l'horizon de prévision. Cette évolution doit toutefois être interprétée avec prudence, les observations disponibles n'étant pas strictement identiques pour chaque horizon.

Ces premiers résultats constituent une référence mesurable pour les expérimentations suivantes.

La prochaine étape consistera à évaluer une **baseline hebdomadaire (H-168)**, fondée sur la consommation observée à la même heure et au même jour de la semaine précédente.

### B - Construction d'une baseline hebdomadaire

La référence hebdomadaire permet de retrouver un jour de la semaine comparable, ce qui pourrait mieux représenter les cycles d'activité.

Nous allons conserver notre DataFrame df_baseline et y ajouter une nouvelle colonne de prédiction.

#### B1 - Construction des prédictions H-168

In [371]:
# Même heure, sept jours avant l'heure cible
df_baseline["weekly_reference_timestamp"] = df_baseline["target_timestamp"] - pd.Timedelta(days=7)

# Recherche des consommations historiques correspondantes
weekly_reference_index = pd.MultiIndex.from_frame(
    df_baseline[["site_id", "weekly_reference_timestamp"]]
)

df_baseline["weekly_prediction"] = hourly_consumption.reindex(weekly_reference_index).to_numpy()

# Vérification de la disponibilité au moment de la prévision
df_baseline["weekly_reference_available"] = (
    df_baseline["weekly_reference_timestamp"] <= df_baseline["forecast_origin"]
) & df_baseline["weekly_prediction"].notna()

print("Disponibilité des références hebdomadaires :")

display(df_baseline.groupby("horizon")["weekly_reference_available"].agg(["sum", "count", "mean"]))

Disponibilité des références hebdomadaires :


,sum,count,mean
horizon,,,
1,1505,1505,1.0
2,1498,1498,1.0
3,1491,1491,1.0
4,1484,1484,1.0
5,1477,1477,1.0
6,1470,1470,1.0
7,1463,1463,1.0
8,1456,1456,1.0
9,1449,1449,1.0


#### B2 - Comparaison des deux baselines

In [372]:
# Observations disposant des deux références
df_comparison = df_baseline.loc[
    df_baseline["reference_available"] & df_baseline["weekly_reference_available"]
].copy()


def evaluate_baseline(data, prediction_column):
    return {
        "MAE": mean_absolute_error(
            data["target_consumption_kw"],
            data[prediction_column],
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                data["target_consumption_kw"],
                data[prediction_column],
            )
        ),
    }


results = pd.DataFrame(
    {
        "Journalière H-24": evaluate_baseline(df_comparison, "baseline_prediction"),
        "Hebdomadaire H-168": evaluate_baseline(df_comparison, "weekly_prediction"),
    }
).T.round(2)

print(f"Observations comparées : {len(df_comparison):,}")
display(results)

Observations comparées : 34,188


,MAE,RMSE
Journalière H-24,36.49,114.25
Hebdomadaire H-168,6.78,10.48


#### B3 - Comparaison par type de site

In [373]:
comparison_by_type = []

for site_type, group in df_comparison.groupby("site_type"):
    for method, prediction_column in [
        ("H-24", "baseline_prediction"),
        ("H-168", "weekly_prediction"),
    ]:
        metrics = evaluate_baseline(group, prediction_column)

        comparison_by_type.append(
            {
                "site_type": site_type,
                "method": method,
                "observations": len(group),
                **metrics,
            }
        )

df_comparison_by_type = pd.DataFrame(comparison_by_type)

display(
    df_comparison_by_type.pivot(
        index="site_type",
        columns="method",
        values=["MAE", "RMSE"],
    ).round(2)
)

MAE          RMSE        
method      H-168   H-24  H-168    H-24
site_type                              
datacenter  13.42  12.22  16.57   15.44
factory      9.72  92.73  13.47  207.28
hospital     7.74   7.98   9.87   10.10
office       1.63  10.66   2.25   26.35
retail       3.62  28.41   4.89   60.97

### Bilan comparatif des baselines

Deux méthodes de référence ont été évaluées sur les 34 188 observations de validation :

- **Persistance journalière (H-24)** : MAE de 36,49 kW et RMSE de 114,25 kW.
- **Persistance hebdomadaire (H-168)** : MAE de 6,78 kW et RMSE de 10,48 kW.

La baseline hebdomadaire réduit l'erreur absolue moyenne d'environ **81,4 %** par rapport à la méthode journalière.

L'analyse par type de site met en évidence des améliorations particulièrement importantes pour les usines, les bureaux et les commerces. Ces résultats confirment l'importance des cycles hebdomadaires dans les profils énergétiques simulés.

À l'inverse, la persistance journalière demeure légèrement plus performante pour les datacenters, dont la consommation présente un comportement davantage continu.

Ces résultats démontrent qu'une méthode de prévision simple, fondée sur la périodicité des consommations, peut constituer une référence particulièrement exigeante pour les modèles de Machine Learning.

Les résultats devront néanmoins être interprétés dans le contexte de données simulées et d'un historique limité à 30 jours.

Pour les expérimentations suivantes, la baseline hebdomadaire constituera notre **principale référence globale**, tandis que les performances par type de site seront également comparées à la meilleure méthode de référence correspondante.

## 5 — Conclusion et perspectives

### A — Synthèse des travaux réalisés

Cette phase d'expérimentation avait pour objectif de définir un cadre méthodologique fiable pour l'entraînement et l'évaluation des futurs modèles de prévision de consommation énergétique.

À partir du dataset supervisé constitué lors de la phase de préparation, nous avons réalisé plusieurs opérations :

- **Séparation chronologique des données** en ensembles d'entraînement et de validation, afin de reproduire une situation de prévision sur des observations futures et de limiter les risques de fuite temporelle.
- **Identification des variables explicatives**, comprenant les caractéristiques des sites, les informations calendaires, les horizons de prévision et les consommations historiques.
- **Construction d'un pipeline de prétraitement** permettant d'encoder les variables catégorielles et de traiter les valeurs manquantes sans exploiter les données de validation pour calculer les paramètres de transformation.
- **Définition de modèles de référence naïfs (*baselines*)**, destinés à mesurer la valeur ajoutée des futurs algorithmes de Machine Learning.

Le dataset couvre sept sites répartis en cinq catégories et permet d'étudier des prévisions allant de **H+1 à H+24**.

La séparation chronologique retenue aboutit à **82 572 exemples d'entraînement** et **34 188 exemples de validation**. Les exemples situés à cheval sur la date de séparation ont été exclus afin de préserver l'intégrité temporelle de l'évaluation.

### B — Résultats des modèles de référence

Deux stratégies de prévision naïves ont été évaluées sur les mêmes observations de validation :

- **Baseline journalière (H-24)** : prédire une consommation identique à celle observée à la même heure la veille.
- **Baseline hebdomadaire (H-168)** : prédire une consommation identique à celle observée à la même heure la semaine précédente.

| Baseline | MAE | RMSE |
|---|---:|---:|
| Journalière — H-24 | 36,49 kW | 114,25 kW |
| Hebdomadaire — H-168 | **6,78 kW** | **10,48 kW** |

La baseline hebdomadaire présente des performances nettement supérieures, avec une réduction d'environ **81 % de la MAE** et **91 % de la RMSE** par rapport à la baseline journalière.

Cette différence suggère une forte régularité hebdomadaire dans les consommations simulées. Elle est particulièrement marquée pour les sites industriels, les bureaux et les commerces, dont les profils d'activité semblent fortement dépendre du jour de la semaine.

Cette tendance n'est toutefois pas uniforme : pour les datacenters, la référence journalière reste légèrement plus performante. Ce constat souligne l'intérêt d'une analyse différenciée selon les catégories de sites.

### C — Enseignements et limites

Plusieurs enseignements peuvent être tirés de ces expérimentations.

**La temporalité constitue un facteur déterminant.** Les consommations historiques ne présentent pas toutes le même pouvoir prédictif. La comparaison des deux baselines montre que la périodicité hebdomadaire est particulièrement informative sur notre jeu de données.

Ce résultat nous a conduits à enrichir les variables explicatives avec `target_lag_168h`, correspondant à la consommation observée sept jours avant l'heure cible. Cette variable est calculée à partir d'informations disponibles au moment de la prévision, afin d'éviter toute fuite de données futures.

**Les comportements énergétiques diffèrent selon les sites.** Les résultats par catégorie montrent qu'une stratégie unique ne sera pas nécessairement optimale pour tous les profils de consommation. Cette observation justifie de comparer, dans les prochaines étapes, une approche globale et des modèles spécialisés par type de site.

**Les résultats doivent néanmoins être interprétés avec prudence.** Les données utilisées sont synthétiques et couvrent une période de trente jours seulement. Elles peuvent présenter des régularités plus marquées que des consommations réelles et ne permettent pas d'étudier correctement les variations saisonnières, les jours fériés ou les changements durables d'activité. Par ailleurs, les exemples issus des différents horizons de prévision se recoupent : ils ne constituent donc pas des observations totalement indépendantes.

Les performances obtenues constituent ainsi des références pour notre preuve de concept, et non une démonstration de la capacité de généralisation du système en conditions réelles.

### D — Perspectives : entraînement et évaluation des modèles

La prochaine étape consistera à entraîner plusieurs algorithmes de Machine Learning sur le dataset préparé.

Nous commencerons par une **régression linéaire**, qui servira de premier modèle supervisé, avant d'expérimenter des approches capables de représenter des relations non linéaires, notamment les forêts aléatoires.

Les modèles seront évalués selon un protocole identique, en conservant la même séparation chronologique et les mêmes métriques, afin de garantir la comparabilité des résultats.

Une attention particulière sera portée :

- Aux performances globales et aux résultats par type de site et par horizon de prévision.
- À la comparaison avec la baseline hebdomadaire, qui constitue désormais notre référence principale.
- À l'apport réel de la variable `target_lag_168h`, notamment en comparant les résultats avec et sans cette information.
- À la traçabilité des expérimentations grâce à MLflow, qui permettra de conserver les paramètres, métriques et versions des modèles.

### Conclusion

Cette phase d'expérimentation nous a permis de passer d'un jeu de données préparé à un **protocole d'apprentissage et d'évaluation structuré et reproductible**.

Elle a également mis en évidence une caractéristique importante de nos données : la forte influence des cycles hebdomadaires sur la consommation énergétique.

L'enjeu de la prochaine phase ne sera donc pas simplement d'obtenir un modèle capable de produire des prévisions, mais de **démontrer qu'il apporte une amélioration mesurable par rapport à une méthode simple, robuste et déjà performante**.

Cette exigence constituera notre principal critère pour sélectionner les modèles à retenir dans la suite du projet Smart Energy Optimizer.